In [2]:
from pathlib import Path
import os, sys, subprocess, zipfile, json, re
from datetime import datetime

# =========================================================
# SETTINGS
# =========================================================
CREATE_DATASET_ZIP = True
RUN_SMOKE_TEST = True
START_BASELINE_TRAIN = False   # change to True after smoke test succeeds
FORCE_REBUILD_ZIP = False
PATCH_STYLEGAN3_TO_REF_IMPL = True

RESOLUTION = 256
MAX_IMAGES_FOR_ZIP = None      # set 5000 for quick debug, keep None for full CelebA
MIRROR = 1
KIMG = 1500
SNAP = 10
METRICS = "none"               # faster first run; later you can use fid50k_full
SEED = 42

# =========================================================
# PATHS
# =========================================================
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

raw_celeba = ROOT / "data" / "raw" / "celeba"
zip_dir = ROOT / "data" / "stylegan3_zip"
zip_dir.mkdir(parents=True, exist_ok=True)

stylegan3_repo = ROOT / "third_party" / "stylegan3"
checkpoints_dir = ROOT / "checkpoints" / "baseline_stylegan3r"
checkpoints_dir.mkdir(parents=True, exist_ok=True)

results_smoke = ROOT / "results" / "generated" / "baseline" / "smoke_test"
results_smoke.mkdir(parents=True, exist_ok=True)

logs_dir = ROOT / "logs" / "baseline"
logs_dir.mkdir(parents=True, exist_ok=True)

cache_dir = ROOT / ".cache"
(cache_dir / "torch_extensions").mkdir(parents=True, exist_ok=True)
(cache_dir / "dnnlib").mkdir(parents=True, exist_ok=True)

dataset_zip = zip_dir / f"celeba_{RESOLUTION}x{RESOLUTION}.zip"

resume_url = "https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/stylegan3-r-ffhqu-256x256.pkl"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Make extension/model caches local to project
os.environ["TORCH_EXTENSIONS_DIR"] = str(cache_dir / "torch_extensions")
os.environ["DNNLIB_CACHE_DIR"] = str(cache_dir / "dnnlib")

# =========================================================
# HELPERS
# =========================================================
def list_images(folder: Path):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

def run_cmd(cmd, cwd=None):
    print("\n[RUN]", " ".join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

def clone_stylegan3_if_needed():
    if stylegan3_repo.exists():
        print(f"[OK] stylegan3 repo exists: {stylegan3_repo}")
        return
    run_cmd(["git", "clone", "https://github.com/nvlabs/stylegan3", str(stylegan3_repo)])

def install_stylegan3_requirements():
    req = stylegan3_repo / "requirements.txt"
    if req.exists():
        run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    else:
        run_cmd([sys.executable, "-m", "pip", "install", "-q",
                 "torch", "torchvision", "click", "requests", "tqdm", "pyspng", "ninja", "imageio", "imageio-ffmpeg"])

def get_gpu_info():
    try:
        import torch
        cuda = torch.cuda.is_available()
        count = torch.cuda.device_count() if cuda else 0
        names = [torch.cuda.get_device_name(i) for i in range(count)] if cuda else []
        return cuda, count, names
    except Exception:
        return False, 0, []

def choose_training_params():
    cuda, gpu_count, gpu_names = get_gpu_info()
    if not cuda or gpu_count == 0:
        raise RuntimeError("No CUDA GPU found. StyleGAN3 training needs GPU.")
    gpus = min(gpu_count, 8)

    # Safe single-GPU batch for 256x256
    if gpus >= 4:
        batch = 32
    elif gpus == 2:
        batch = 16
    else:
        batch = 8

    gamma = 6.6
    return {
        "cuda": cuda,
        "gpu_count_visible": gpu_count,
        "gpu_names": gpu_names,
        "gpus_used": gpus,
        "batch": batch,
        "gamma": gamma,
    }

def build_dataset_zip():
    assert raw_celeba.exists(), f"Missing raw CelebA folder: {raw_celeba}"
    n_raw = len(list_images(raw_celeba))
    assert n_raw > 0, f"No images found in {raw_celeba}"

    if dataset_zip.exists() and not FORCE_REBUILD_ZIP:
        print(f"[SKIP] Dataset ZIP already exists: {dataset_zip}")
        return

    dataset_tool = stylegan3_repo / "dataset_tool.py"
    assert dataset_tool.exists(), f"Missing dataset_tool.py in {stylegan3_repo}"

    cmd = [
        sys.executable, str(dataset_tool),
        "--source", str(raw_celeba),
        "--dest", str(dataset_zip),
        "--resolution", f"{RESOLUTION}x{RESOLUTION}",
        "--transform", "center-crop",
    ]
    if MAX_IMAGES_FOR_ZIP is not None:
        cmd += ["--max-images", str(MAX_IMAGES_FOR_ZIP)]

    run_cmd(cmd, cwd=stylegan3_repo)

def inspect_zip():
    assert dataset_zip.exists(), f"ZIP not found: {dataset_zip}"
    with zipfile.ZipFile(dataset_zip, "r") as zf:
        names = zf.namelist()
        pngs = [n for n in names if n.lower().endswith(".png")]
        has_json = "dataset.json" in names
        print(f"ZIP = {dataset_zip}")
        print(f"PNG count inside ZIP = {len(pngs)}")
        print(f"Has dataset.json = {has_json}")
        assert len(pngs) > 0, "No PNG images inside dataset ZIP."
        assert has_json, "dataset.json missing inside dataset ZIP."

def replace_once(path: Path, old: str, new: str):
    text = path.read_text(encoding="utf-8")
    if old not in text:
        return False
    text = text.replace(old, new, 1)
    path.write_text(text, encoding="utf-8")
    return True

def patch_stylegan3_to_ref_impl():
    """
    Force StyleGAN3 to use Python/reference ops instead of custom CUDA plugins.
    This avoids bias_act_plugin / upfirdn2d_plugin / filtered_lrelu_plugin compilation.
    """
    bias_act_py = stylegan3_repo / "torch_utils" / "ops" / "bias_act.py"
    upfirdn2d_py = stylegan3_repo / "torch_utils" / "ops" / "upfirdn2d.py"
    filtered_lrelu_py = stylegan3_repo / "torch_utils" / "ops" / "filtered_lrelu.py"

    assert bias_act_py.exists(), f"Missing {bias_act_py}"
    assert upfirdn2d_py.exists(), f"Missing {upfirdn2d_py}"
    assert filtered_lrelu_py.exists(), f"Missing {filtered_lrelu_py}"

    patched = {}

    patched["bias_act"] = replace_once(
        bias_act_py,
        "def bias_act(x, b=None, dim=1, act='linear', alpha=None, gain=None, clamp=None, impl='cuda'):",
        "def bias_act(x, b=None, dim=1, act='linear', alpha=None, gain=None, clamp=None, impl='ref'):"
    )

    patched["upfirdn2d"] = replace_once(
        upfirdn2d_py,
        "def upfirdn2d(x, f, up=1, down=1, padding=0, flip_filter=False, gain=1, impl='cuda'):",
        "def upfirdn2d(x, f, up=1, down=1, padding=0, flip_filter=False, gain=1, impl='ref'):"
    )

    patched["filtered_lrelu"] = replace_once(
        filtered_lrelu_py,
        "def filtered_lrelu(x, fu=None, fd=None, b=None, up=1, down=1, padding=0, gain=np.sqrt(2), slope=0.2, clamp=None, flip_filter=False, impl='cuda'):",
        "def filtered_lrelu(x, fu=None, fd=None, b=None, up=1, down=1, padding=0, gain=np.sqrt(2), slope=0.2, clamp=None, flip_filter=False, impl='ref'):"
    )

    patch_report = logs_dir / "stylegan3_ref_patch_report.json"
    patch_report.write_text(json.dumps(patched, indent=2), encoding="utf-8")
    print("Patch report:", json.dumps(patched, indent=2))
    print("Saved patch report to:", patch_report)

def run_smoke_test():
    gen_images = stylegan3_repo / "gen_images.py"
    assert gen_images.exists(), f"Missing gen_images.py in {stylegan3_repo}"
    cmd = [
        sys.executable, str(gen_images),
        "--outdir", str(results_smoke),
        "--trunc", "0.7",
        "--seeds", "2,5,9,15",
        "--network", resume_url,
    ]
    run_cmd(cmd, cwd=stylegan3_repo)

def latest_run_folder(outdir: Path):
    if not outdir.exists():
        return None
    runs = [p for p in outdir.iterdir() if p.is_dir()]
    if not runs:
        return None
    runs = sorted(runs, key=lambda p: p.stat().st_mtime, reverse=True)
    return runs[0]

# =========================================================
# START
# =========================================================
print("ROOT =", ROOT)
print("Time =", datetime.now().isoformat())
print("TORCH_EXTENSIONS_DIR =", os.environ["TORCH_EXTENSIONS_DIR"])
print("DNNLIB_CACHE_DIR =", os.environ["DNNLIB_CACHE_DIR"])

raw_count = len(list_images(raw_celeba))
print("Raw CelebA images found =", raw_count)
assert raw_count > 0, "Raw CelebA folder is empty. Notebook 02 must complete first."

clone_stylegan3_if_needed()
install_stylegan3_requirements()

gpu_info = choose_training_params()
print(json.dumps(gpu_info, indent=2))

if CREATE_DATASET_ZIP:
    build_dataset_zip()
    inspect_zip()

if PATCH_STYLEGAN3_TO_REF_IMPL:
    patch_stylegan3_to_ref_impl()

if RUN_SMOKE_TEST:
    run_smoke_test()
    smoke_imgs = sorted(results_smoke.glob("*.png"))
    print(f"Smoke test images created = {len(smoke_imgs)}")
    for p in smoke_imgs[:8]:
        print("  ", p)

train_py = stylegan3_repo / "train.py"
assert train_py.exists(), f"Missing train.py in {stylegan3_repo}"

train_cmd = [
    sys.executable, str(train_py),
    "--outdir", str(checkpoints_dir),
    "--cfg", "stylegan3-r",
    "--data", str(dataset_zip),
    "--gpus", str(gpu_info["gpus_used"]),
    "--batch", str(gpu_info["batch"]),
    "--gamma", str(gpu_info["gamma"]),
    "--mirror", str(MIRROR),
    "--kimg", str(KIMG),
    "--snap", str(SNAP),
    "--metrics", str(METRICS),
    "--seed", str(SEED),
    "--resume", resume_url,
]

command_txt = logs_dir / "baseline_train_command.txt"
command_txt.write_text(" ".join(str(x) for x in train_cmd), encoding="utf-8")
print("\nSaved training command to:", command_txt)
print("Training command:\n")
print(" ".join(str(x) for x in train_cmd))

if START_BASELINE_TRAIN:
    run_cmd(train_cmd, cwd=stylegan3_repo)
    latest = latest_run_folder(checkpoints_dir)
    print("\nLatest training run folder:", latest)
else:
    print("\nSTART_BASELINE_TRAIN is False, so training was not launched.")
    print("If smoke test works now, set START_BASELINE_TRAIN = True and run this notebook again.")

print("\nNotebook 04 complete.")
print("Next step: once smoke test succeeds, launch baseline training here.")

ROOT = /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation
Time = 2026-04-07T14:38:14.955003
TORCH_EXTENSIONS_DIR = /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/.cache/torch_extensions
DNNLIB_CACHE_DIR = /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/.cache/dnnlib
Raw CelebA images found = 202599
[OK] stylegan3 repo exists: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/third_party/stylegan3

[RUN] /home/sajjan/.conda/envs/myenv/bin/python -m pip install -q torch torchvision click requests tqdm pyspng ninja imageio imageio-ffmpeg
{
  "cuda": true,
  "gpu_count_visible": 1,
  "gpu_names": [
    "NVIDIA H100 NVL"
  ],
  "gpus_used": 1,
  "batch": 8,
  "gamma": 6.6
}
[SKIP] Dataset ZIP already exists: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/data/stylegan3_zip/celeba_256x256.zip
ZIP = /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/data/stylegan3_zip/celeba_256x256.zip
PNG count inside ZIP = 202599

/data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/third_party/stylegan3/torch_utils/ops/conv2d_gradfix.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Generating image for seed 2 (0/4) ...
Generating image for seed 5 (1/4) ...
Generating image for seed 9 (2/4) ...
Generating image for seed 15 (3/4) ...
Smoke test images created = 4
   /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/results/generated/baseline/smoke_test/seed0002.png
   /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/results/generated/baseline/smoke_test/seed0005.png
   /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/results/generated/baseline/smoke_test/seed0009.png
   /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/results/generated/baseline/smoke_test/seed0015.png

Saved training command to: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/logs/baseline/baseline_train_command.txt
Training command:

/home/sajjan/.conda/envs/myenv/bin/python /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/third_party/stylegan3/train.py --outdir /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generati